In [9]:
### This notebook is used for testing the correct implementation of new models in the pipeline

In [10]:
ROOT = '/home/nicolo_b/Desktop/PhD/RELIABLE_NAS/NOTEBOOK/FAULT_INJECTOR/VITTFI'
DATA_ROOT = f'{ROOT}/data'
NETWORK_NAME = 'RESNET56_SGD_EP_200_LR_01'
DATASET = 'CIFAR100'
WEIGHTS_PATH = f'{ROOT}/models/{DATASET}/pretrained/{NETWORK_NAME}_{DATASET}.pt'
BATCH_SIZE = 128

In [11]:
import torch
import sys
sys.path.insert(0, ROOT)
import importlib, importlib.util
spec = importlib.util.spec_from_file_location("nicolo_net", f"{ROOT}/models/nicolo_net.py")
nicolo_net = importlib.util.module_from_spec(spec)
spec.loader.exec_module(nicolo_net)
from utils import get_network, get_loader

In [12]:
# Controlla tipo e chiavi del checkpoint
ckpt = torch.load(WEIGHTS_PATH, map_location='cpu', weights_only=True)
print(type(ckpt))
print(ckpt.keys() if isinstance(ckpt, dict) else 'è direttamente uno state dict')

<class 'collections.OrderedDict'>
odict_keys(['conv1.weight', 'bn1.weight', 'bn1.bias', 'bn1.running_mean', 'bn1.running_var', 'bn1.num_batches_tracked', 'layer1.0.conv1.weight', 'layer1.0.bn1.weight', 'layer1.0.bn1.bias', 'layer1.0.bn1.running_mean', 'layer1.0.bn1.running_var', 'layer1.0.bn1.num_batches_tracked', 'layer1.0.conv2.weight', 'layer1.0.bn2.weight', 'layer1.0.bn2.bias', 'layer1.0.bn2.running_mean', 'layer1.0.bn2.running_var', 'layer1.0.bn2.num_batches_tracked', 'layer1.1.conv1.weight', 'layer1.1.bn1.weight', 'layer1.1.bn1.bias', 'layer1.1.bn1.running_mean', 'layer1.1.bn1.running_var', 'layer1.1.bn1.num_batches_tracked', 'layer1.1.conv2.weight', 'layer1.1.bn2.weight', 'layer1.1.bn2.bias', 'layer1.1.bn2.running_mean', 'layer1.1.bn2.running_var', 'layer1.1.bn2.num_batches_tracked', 'layer1.2.conv1.weight', 'layer1.2.bn1.weight', 'layer1.2.bn1.bias', 'layer1.2.bn1.running_mean', 'layer1.2.bn1.running_var', 'layer1.2.bn1.num_batches_tracked', 'layer1.2.conv2.weight', 'layer1.2.b

In [13]:
# Carica la rete tramite get_network e verifica network[-1] e forward pass
network = get_network(
    network_name=NETWORK_NAME,
    device=torch.device('cpu'),
    dataset_name=DATASET,
    root=ROOT
)
print(network[-1])
x = torch.randn(1, 3, 32, 32)
out = network(x)
print(out.shape)

Loading network RESNET56_SGD_EP_200_LR_01
Linear(in_features=64, out_features=100, bias=True)
torch.Size([1, 100])


In [14]:
# Valuta accuracy sul test set tramite get_loader
train_loader, test_loader = get_loader(
    network_name=NETWORK_NAME,
    batch_size=BATCH_SIZE,
    dataset_name=DATASET,
    root=DATA_ROOT
)

network.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        outputs = network(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
print(f'Accuracy: {100 * correct / total:.2f}%')

Loading CIFAR100 dataset
Dataset loaded
Batch size:		128 
Number of batches:	79
Accuracy: 72.61%
